In [1]:
import os, json, shutil, hashlib, zipfile, datetime
import requests
from glob import glob
BASE = '/content/drive/MyDrive/wpl' # Define base directory (e.g. '.' or '/content/drive/MyDrive' if using Google Drive)
SOURCE_URL   = 'https://cricsheet.org/downloads/wpl_json.zip'
SNAPSHOT_DIR = f'{BASE}/data/snapshots'                  # on Drive: survives Colab restarts
os.makedirs(SNAPSHOT_DIR, exist_ok=True)
SNAPSHOT = os.path.join(SNAPSHOT_DIR, 'wpl_json_20261025.zip')
if not os.path.exists(SNAPSHOT):
    SNAPSHOT = 'download'
REFERENCE_SHA = '533e7937056b47f1'                       # the stage 1 reference data
DATA_DIR     = 'wpl_data'
def sha256_of_file(path, chunk_size=1 << 20):
    """SHA-256 of the file's bytes as stored on disk, read in chunks."""
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk_size), b''):
            h.update(block)
    return h.hexdigest()
def manifest_path_for(zip_path):
  return zip_path[:-len('.zip')] + '.manifest.json'
if SNAPSHOT == 'download':
    os.makedirs(SNAPSHOT_DIR, exist_ok=True)
    now = datetime.datetime.now(datetime.timezone.utc)
    zip_path = os.path.join(SNAPSHOT_DIR, f'wpl_json_{now:%Y%m%d}.zip')
    tmp_path = zip_path + '.part'

    resp = requests.get(SOURCE_URL, timeout=120)
    resp.raise_for_status()                      # fail loudly on 4xx/5xx

    with open(tmp_path, 'wb') as f:              # file is closed (fully written)
        f.write(resp.content)                    # when this block exits

    if not zipfile.is_zipfile(tmp_path):
        os.remove(tmp_path)
        raise ValueError(f'{SOURCE_URL} did not return a valid zip file')

    new_sha = sha256_of_file(tmp_path)
    if os.path.exists(zip_path):
        if sha256_of_file(zip_path) != new_sha:
            os.remove(tmp_path)
            raise RuntimeError(
                f'{zip_path} already exists with different contents. Cricsheet changed '
                'the file today. Rename or move the old snapshot before downloading again.'
            )
        os.remove(tmp_path)                      # identical, keep the existing one
    else:
        os.replace(tmp_path, zip_path)           # atomic rename, no half-written snapshot

    source, downloaded_at = SOURCE_URL, now.isoformat()
else:
    zip_path = SNAPSHOT
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f'Snapshot not found: {zip_path}')
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f'Not a valid zip file: {zip_path}')
    source, downloaded_at = None, None
SNAPSHOT_SHA = sha256_of_file(zip_path)

# ---- 4. Manifest (the receipt) ----------------------------------------
manifest_path = manifest_path_for(zip_path)
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        existing = json.load(f)
    if existing.get('sha256') != SNAPSHOT_SHA:
        raise RuntimeError(f'{zip_path} does not match the hash in {manifest_path}')
else:
    manifest = {
        'file': os.path.basename(zip_path),
        'source_url': source,                   # None if the file came from somewhere else
        'sha256': SNAPSHOT_SHA,
        'downloaded_at_utc': downloaded_at,     # None if not downloaded by this cell
    }
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

# ---- 5. Fresh extraction, no leftovers --------------------------------
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(DATA_DIR)

# ---- 6. Report ---------------------------------------------------------
n_json = len(glob(os.path.join(DATA_DIR, '*.json')))
nested = [os.path.join(r, n) for r, _, fs in os.walk(DATA_DIR) if r != DATA_DIR for n in fs if n.endswith('.json')]
if nested:
    raise RuntimeError(f'{len(nested)} .json files are in subfolders; the loaders only read {DATA_DIR}/*.json')

print(f'Snapshot : {zip_path}')
print(f'SHA-256  : {SNAPSHOT_SHA[:16]}...')
print(f'JSON files in {DATA_DIR}/: {n_json}')
assert SNAPSHOT_SHA.startswith(REFERENCE_SHA), f'Data changed: {SNAPSHOT_SHA[:16]} is not the stage 1 reference'

Snapshot : ./data/snapshots/wpl_json_20261025.zip
SHA-256  : 533e7937056b47f1...
JSON files in wpl_data/: 88


In [2]:
import json
import os
from glob import glob

import pandas as pd

TEAM_ALIASES = {"Royal Challengers Bangalore": "Royal Challengers Bengaluru"}


def canon(team):
    """Canonical team name, so a renamed franchise keeps one history."""
    return TEAM_ALIASES.get(team, team)


def resolve_winner(info):
    """Label rules: listed winner; super-over tie -> super-over winner;
    no result / tie without super over -> None (match excluded)."""
    outcome = info.get('outcome', {})
    if 'winner' in outcome:
        return outcome['winner'], 'win'
    if outcome.get('eliminator'):
        return outcome['eliminator'], 'tie_super_over'
    return None, None


def load_and_extract_match_data(data_folder="wpl_data"):
    all_files = sorted(glob(os.path.join(data_folder, "*.json")))
    if not all_files:
        raise FileNotFoundError(f"No .json files in {data_folder}/ - run the snapshot cell first")

    extracted_data = []
    skipped = []
    for file in all_files:
        with open(file, 'r', encoding='utf-8') as f:
            match = json.load(f)
        match_id = os.path.basename(file)[:-5]
        info = match['info']

        winner, result_type = resolve_winner(info)
        if winner is None:
            skipped.append((match_id, info.get('outcome', {})))
            continue

        main_innings = [inn for inn in match['innings'] if not inn.get('super_over')]
        bat_first = main_innings[0]['team']                       # raw name
        chase = [t for t in info['teams'] if t != bat_first][0]   # raw name
        y = int(winner == bat_first)                              # compare raw names

        registry = info.get('registry', {}).get('people', {})
        replacements = [r for inn in match['innings'] for o in inn['overs']
                        for d in o['deliveries'] for r in d.get('replacements', {}).get('match', [])]

        extracted_data.append({
            'match_id': match_id,
            'date': pd.Timestamp(info['dates'][0]),
            'season': int(info['dates'][0][:4]),
            'venue': info['venue'],
            'bat_first_team': canon(bat_first),
            'chase_team': canon(chase),
            'toss_winner': canon(info['toss']['winner']),
            'toss_decision': info['toss']['decision'],
            'winner': canon(winner),
            'result_type': result_type,
            'y': y,                                               # 1 = team batting first won
            'bat_first_players': info['players'][bat_first],
            'chase_players': info['players'][chase],
            'bat_first_player_ids': [registry.get(p) for p in info['players'][bat_first]],
            'chase_player_ids': [registry.get(p) for p in info['players'][chase]],
            'replacements': replacements,
        })

    if skipped:
        print(f"Skipped {len(skipped)} matches with no winner:", skipped)
    return pd.DataFrame(extracted_data).sort_values(['date', 'match_id']).reset_index(drop=True)

In [3]:
WINDOW = 30
def innings_extract(deliveries, window=30):
    rows = []
    runs = wickets = legal = 0
    runs_at_legal = [0]      # cumulative runs when legal ball k was completed (k = 0: start)
    wkts_at_legal = [0]
    part_runs0 = part_legal0 = 0   # score when the current partnership began
    for d in deliveries:
        runs += int(d['runs'])
        legal += int(bool(d['legal']))
        wickets = min(wickets + int(d['wickets']), 10)
        if d['legal']:
            runs_at_legal.append(runs)
            wkts_at_legal.append(wickets)
        if d['wickets']:                      # a new partnership starts after a dismissal
            part_runs0, part_legal0 = runs, legal
        k = max(legal - window, 0)            # window = everything after legal ball k
        rows.append({
            'delivery_runs': int(d['runs']),
            'is_legal': int(bool(d['legal'])),
            'is_wicket': int(d['wickets'] > 0),
            'cum_runs': runs,
            'cum_wickets': wickets,
            'legal_balls': legal,
            'runs_last_30': runs - runs_at_legal[k],
            'wkts_last_30': wickets - wkts_at_legal[k],
            'partnership_runs': runs - part_runs0,
            'partnership_balls': legal - part_legal0,
        })
    return rows

In [4]:
##ball by ball data
import numpy as np

NOT_DISMISSALS = {'retired hurt', 'retired not out'}
def overs_to_balls(overs):
    return int(overs)*6+ round((overs%1)*10)

def load_ball_by_ball_data(data_folder ='wpl_data'): # Changed default data_folder
  files = sorted(glob(os.path.join(data_folder, "*.json")))
  all_balls_records = [] # Initialized to accumulate all ball records

  for file in files:
      try:
          with open(file, 'r', encoding='utf-8') as f:
              match = json.load(f)

          info = match['info']
          match_id = os.path.basename(file)[:-5]
          winner, result_type = resolve_winner(info)   # same label rule as cell 0
          if winner is None:
              continue

          main = [inn for inn in match['innings'] if not inn.get('super_over')][:2]
          bat_first = main[0]['team']
          y = int(winner == bat_first)                  # label: team batting first won

          # target and balls available, read once per match
          inn1_total = sum(d['runs']['total'] for o in main[0]['overs'] for d in o['deliveries'])
          tgt = main[1].get('target', {}) if len(main) > 1 else {}
          target = tgt.get('runs', inn1_total + 1)
          sched_balls = overs_to_balls(info.get('overs', 20))
          max_balls_by_innings = {1: sched_balls,
                                  2: overs_to_balls(tgt['overs']) if 'overs' in tgt else sched_balls}

          for innings_idx, innings_dict in enumerate(main):
            batting_team = innings_dict['team']
            innings_num = innings_idx + 1 # 1-based indexing for innings
            bowling_team = [t for t in info['teams'] if t != batting_team][0]
            max_balls = max_balls_by_innings[innings_num]


                       # 1) the innings as a plain list of deliveries: the same shape the API receives
            deliveries, positions = [], []
            for over_dict in innings_dict['overs']:
              for ball_num, delivery in enumerate(over_dict['deliveries']):
                extras = delivery.get('extras', {})
                deliveries.append({
                    'runs': delivery['runs']['total'],
                    'legal': 'wides' not in extras and 'noballs' not in extras,
                    'wickets': sum(1 for w in delivery.get('wickets', []) if w['kind'] not in NOT_DISMISSALS),
                })
                positions.append((over_dict['over'], ball_num))

            # 2) state after every delivery, from the shared function
            for seq, (state, (over_num, ball_num)) in enumerate(zip(innings_state(deliveries), positions), 1):
                all_balls_records.append({
                    'match_id': match_id,
                    'y': y,
                    'venue': info['venue'],
                    'date': pd.Timestamp(info['dates'][0]),
                    'season': int(info['dates'][0][:4]),
                    'innings': innings_num,
                    'batting_team': canon(batting_team),
                    'bowling_team': canon(bowling_team),
                    'max_balls': max_balls,
                    'target': target if innings_num == 2 else np.nan,
                    'over_num': over_num,
                    'ball_num': ball_num,
                    'seq': seq,
                    **state,
                })
      except Exception as e:
          print(f"Error loading or processing file {file}: {e}")

  if not all_balls_records:
      raise ValueError(f"No ball rows extracted from {data_folder}/")

  balls_data_df = pd.DataFrame(all_balls_records)
  return balls_data_df.sort_values(['date', 'match_id', 'innings', 'seq']).reset_index(drop=True)

In [5]:
from collections import deque, defaultdict
FORM_WINDOW= 10
PREMATCH_FEATURES = ['form_diff', 'h2h_adv', 'toss_bf']

class TeamHistory():
  def __init__(self,window = FORM_WINDOW):
    self.window = window
    self.results = defaultdict(lambda: deque(maxlen=window))
    self.h2h = defaultdict(lambda :[0,0])
  def _form(self, team):
    r = self.results[team]
    return (sum(r) + 1) / (len(r) + 2)
  def features(self,bat_first,chase,toss_bf):
    wins,games = self.h2h.get((bat_first,chase) , (0,0))
    return {'form_diff': self._form(bat_first) - self._form(chase),
                'h2h_adv': (wins + 1) / (games + 2) - 0.5,
                'toss_bf': int(toss_bf)}
  def update(self,bat_first,chase,bat_first_won):
    w = int(bat_first_won)
    self.results[bat_first].append(w)
    self.results[chase].append(1-w)
    self.h2h[(bat_first,chase)][0] += w
    self.h2h[(bat_first,chase)][1] += 1
    self.h2h[(chase, bat_first)][0] += 1 - w
    self.h2h[(chase, bat_first)][1] += 1
  def to_dict(self):
        """Plain lists and dicts only, so the model bundle loads anywhere."""
        return {'window': self.window,
                'results': {team: list(r) for team, r in self.results.items()},
                'h2h': [[a, b, w, g] for (a, b), (w, g) in self.h2h.items()]}
  @property
  def teams(self):
    return sorted(self.results)
def build_match_features(match_df):
    """One row per match in date order; features use strictly earlier matches only."""
    history = TeamHistory()
    rows = []
    for _, m in match_df.sort_values(['date', 'match_id']).iterrows():
        toss_bf = int(m['toss_winner'] == m['bat_first_team'])     # toss winner chose to bat
        feats = history.features(m['bat_first_team'], m['chase_team'], toss_bf)   # 1) read
        rows.append({'match_id': m['match_id'], 'season': m['season'], 'date': m['date'],
                     **feats, 'y': int(m['y'])})
        history.update(m['bat_first_team'], m['chase_team'], m['y'])            # 2) then update
    return pd.DataFrame(rows), history

In [6]:
import numpy as np
import pandas as pd

def live_features_add(df):
    balls_df = df.sort_values(['date', 'match_id', 'innings', 'seq']).reset_index(drop=True)

    # Basic calculations
    balls_df['current_rr'] = (balls_df['cum_runs'] / (balls_df['legal_balls'].replace(0, np.nan) / 6)).fillna(0)
    balls_df['balls_remaining'] = (balls_df['max_balls'] - balls_df['legal_balls']).clip(lower=0)
    balls_df['overs_remaining'] = balls_df['balls_remaining'] / 6
    balls_df['wickets_left'] = 10 - balls_df['cum_wickets']

    # Innings 2 targets
    RRR_CAP = 36.0
    inn2 = balls_df['innings'] == 2
    balls_df['target'] = balls_df['target'].where(inn2, 0.0)
    balls_df['runs_needed'] = (balls_df['target'] - balls_df['cum_runs']).where(inn2, 0.0)
    rrr = np.where(balls_df['balls_remaining'] > 0,
                   balls_df['runs_needed'] * 6 / balls_df['balls_remaining'].replace(0, np.nan),
                   np.where(balls_df['runs_needed'] > 0, RRR_CAP, 0.0))
    balls_df['required_rr'] = np.clip(np.nan_to_num(rrr), 0.0, RRR_CAP)
    balls_df['rr_diff'] = (balls_df['required_rr'] - balls_df['current_rr']).where(inn2, 0.0)
    balls_df['projected_total'] = balls_df['cum_runs'] + balls_df['current_rr'] * balls_df['balls_remaining'] / 6

    balls_df['resources_remaining'] = ((balls_df['balls_remaining'] / balls_df['max_balls']) * (balls_df['wickets_left'] / 10))

    # Phase flags
    balls_df['is_powerplay'] = (balls_df['over_num'] < 6).astype(int)
    balls_df['is_death_overs'] = (balls_df['over_num'] >= 16).astype(int)
    balls_df['is_middle_overs'] = ((balls_df['over_num'] >= 6) & (balls_df['over_num'] < 16)).astype(int)



    return balls_df

In [7]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np


In [8]:
def make_prematch():
  return Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(C=1.0, max_iter=2000))])
def train_prematch_model(table):
  oof= []
  for train_seasons, test_season in season_folds(table['season'].unique()):
    tr = table[table['season'].isin(train_seasons)]
    te = table[table['season'] == test_season]
    assert not set(tr['match_id']) & set(te['match_id']), "match in both train and test"
    out = te[['match_id', 'season', 'y']].copy()
    out['base_rate'] = tr['y'].mean()
    model = make_prematch()
    model.fit(tr[PREMATCH_FEATURES], tr['y'])
    out['logistic'] = model.predict_proba(te[PREMATCH_FEATURES])[:, 1]
    oof.append(out)
    print(f"fold: train {train_seasons} -> test {test_season} ({len(te)} matches)")
    # Final model on ALL matches; the evaluation step decides whether it is deployed
  final = make_prematch().fit(table[PREMATCH_FEATURES], table['y'])
  return {'oof': pd.concat(oof, ignore_index=True), 'final': final,
          'p_base': float(table['y'].mean()), 'features': PREMATCH_FEATURES}

In [9]:
### During match prediction
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# One feature list per innings: only innings 2 knows the target.
MOMENTUM = ['runs_last_30', 'wkts_last_30', 'partnership_runs', 'partnership_balls']
FEATURES = {
    1: ['cum_runs', 'wickets_left', 'balls_remaining', 'current_rr', 'projected_total'] + MOMENTUM,
    2: ['runs_needed', 'wickets_left', 'balls_remaining', 'required_rr', 'current_rr', 'target'] + MOMENTUM,
}


def make_logistic():
    # the scaler lives INSIDE the pipeline, so it is saved with the model
    return Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(C=1.0, max_iter=2000))])


def make_rf():
    # legacy settings minus class_weight='balanced' (it distorted the probabilities)
    return RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_split=50,
                                  random_state=42, n_jobs=-1)


MODELS = {'logistic': make_logistic, 'rf': make_rf}


def terminal_prob(df):
    """P(team batting first wins) when the chase is already decided, else NaN."""
    out = pd.Series(np.nan, index=df.index)
    inn2 = df['innings'] == 2
    need = df['target'] - df['cum_runs']
    done = inn2 & ((df['legal_balls'] >= df['max_balls']) | (df['cum_wickets'] >= 10))
    out[inn2 & (need <= 0)] = 0.0      # chaser reached the target
    out[done & (need == 1)] = 0.5      # scores level: super over, treated as a coin flip
    out[done & (need >= 2)] = 1.0      # chaser ran out of balls or wickets
    return out


def fit_per_innings(df, factory):
    models = {}
    for inn in (1, 2):
        sub = df[df['innings'] == inn]
        X = sub[FEATURES[inn]]
        assert not X.isna().any().any(), f"NaN in innings-{inn} features"
        models[inn] = factory().fit(X, sub['y'])
    return models


def predict_per_innings(models, df):
    p = np.empty(len(df))
    for inn in (1, 2):
        m = (df['innings'] == inn).values
        if m.any():
            p[m] = models[inn].predict_proba(df.loc[m, FEATURES[inn]])[:, 1]
    term = terminal_prob(df).values
    return np.where(np.isnan(term), p, term)


def season_folds(seasons, min_train_seasons=1):
    """Rolling origin: train on every season before S, test on S."""
    s = sorted(set(int(x) for x in seasons))
    return [(s[:i], s[i]) for i in range(min_train_seasons, len(s))]


def train_live_model(df):
    oof = []
    for train_seasons, test_season in season_folds(df['season'].unique()):
        tr = df[df['season'].isin(train_seasons)]
        te = df[df['season'] == test_season]
        assert not set(tr['match_id']) & set(te['match_id']), "match in both train and test"
        out = te[['match_id', 'season', 'innings', 'over_num', 'seq', 'y']].copy()
        out['base_rate'] = tr.groupby('match_id')['y'].first().mean()   # match-level share
        for name, factory in MODELS.items():
            out[name] = predict_per_innings(fit_per_innings(tr, factory), te)
        oof.append(out)
        print(f"fold: train {train_seasons} -> test {test_season} "
              f"({te['match_id'].nunique()} matches, {len(te)} rows)")
    oof = pd.concat(oof, ignore_index=True)

    # Final models on ALL seasons. Which one is deployed is decided in the
    # evaluation cell by the pre-registered rule, never here.
    final = {name: fit_per_innings(df, factory) for name, factory in MODELS.items()}
    return {'oof': oof, 'final': final, 'features': FEATURES}

In [10]:
class CricApiClient:
  def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://api.cricapi.com/v1"

  def get_current_matches(self):
        """Get list of ongoing matches"""
        url = f"{self.base_url}/currentMatches?apikey={self.api_key}"

        try:
            response = requests.get(url, timeout=10)
            data = response.json()

            if data.get('status') == 'success':
                return data.get('data', [])
            else:
                print(f"API Error: {data.get('status')}")
                return []
        except Exception as e:
            print(f"Request failed: {e}")
            return []

  def get_match_info(self, match_id):
        """Get detailed match information"""
        url = f"{self.base_url}/match_info?apikey={self.api_key}&id={match_id}"

        try:
            response = requests.get(url, timeout=10)
            data = response.json()

            if data.get('status') == 'success':
                return data.get('data', {})
            return None
        except Exception as e:
            print(f"Request failed: {e}")
            return None

  def get_live_score(self, match_id):
        """Get current live score"""
        match_info = self.get_match_info(match_id)

        if not match_info:
            return None

        # Extract current state
        score_data = match_info.get('score', [])

        if not score_data:
          return None # Handle case where score_data is empty

        current_innings = score_data[-1]

        return {
            'runs': current_innings.get('r', 0),
            'wickets': current_innings.get('w', 0),
            'overs': current_innings.get('o', 0),
            'innings': len(score_data),
            'batting_team': current_innings.get('inning', ''),
            'match_status': match_info.get('status', ''),
            'teams': match_info.get('teams', [])
        }

  def convert_to_model_state(self, live_data, target=None):
        """Convert API data to model input format"""
        overs = live_data['overs']
        balls_faced = int(overs * 6)
        state = {
        'innings': live_data['innings'],
        'cum_runs': live_data['runs'],
        'cum_wickets': live_data['wickets'],
        'wickets_left': 10 - live_data['wickets'],
        'balls_faced': balls_faced,
        'balls_remaining': 120 - balls_faced,
        'overs_remaining': (120 - balls_faced) / 6,
        'current_rr': live_data['runs'] / overs if overs > 0 else 0,
        'is_powerplay': 0 if overs >= 6 else 1,
        'is_middle_overs': 1 if 6 <= overs < 16 else 0,
        'is_death_overs': 1 if overs >= 16 else 0,
        'resources_remaining': ((120 - balls_faced) / 120) * ((10 - live_data['wickets']) / 10),
        'runs_last_6_overs': 0,
        'target': target or 0,
        'runs_needed': 0,
        'required_rr': 0,
        'rr_diff': 0,
        'run_rate_pressure': 0
        }

        # Calculate 2nd innings metrics
        if live_data['innings'] == 2 and target:
            state['target'] = target
            state['runs_needed'] = target - live_data['runs']
            state['required_rr'] = (state['runs_needed'] / state['overs_remaining']) if state['overs_remaining'] > 0 else 0
            state['rr_diff'] = state['required_rr'] - state['current_rr']
            state['run_rate_pressure'] = state['rr_diff'] * (1 - state['resources_remaining'])

        return state

In [11]:
innings_state = innings_extract
match_df = load_and_extract_match_data(DATA_DIR)
ball_df = live_features_add(load_ball_by_ball_data(DATA_DIR))
live_result = train_live_model(ball_df)


fold: train [2023] -> test 2024 (22 matches, 5177 rows)
fold: train [2023, 2024] -> test 2025 (22 matches, 5216 rows)
fold: train [2023, 2024, 2025] -> test 2026 (22 matches, 5250 rows)


In [12]:
# 1. label: one value per match, equal to the match table
print('label ok:', ball_df.groupby('match_id').y.nunique().max() == 1,
      (ball_df.groupby('match_id').y.first() == match_df.set_index('match_id').y).all())
# 2. check B: what the fresh data contains
print(len(match_df), match_df.result_type.value_counts().to_dict(),
      match_df.groupby('season').size().to_dict(), round(match_df.y.mean(), 3))
print(len(ball_df), ball_df.groupby('innings').size().to_dict(),
      'inn2 missing target:', ball_df.loc[ball_df.innings == 2, 'target'].isna().sum(),
      'reduced-overs innings:', (ball_df.groupby(['match_id', 'innings']).max_balls.first() < 120).sum())
# 3. momentum window is 30 LEGAL balls
g = ball_df[(ball_df.match_id == ball_df.match_id.iloc[0]) & (ball_df.innings == 1)].sort_values('seq')
r = g.iloc[-1]; k = r.legal_balls - 30
base = g[(g.legal_balls == k) & (g.is_legal == 1)].cum_runs.iloc[0] if k > 0 else 0
print('window ok:', r.runs_last_30 == r.cum_runs - base)
# 4. no stale columns
print('stale columns:', [c for c in ('batting_team_won', 'runs_last_6_overs') if c in ball_df])

label ok: True True
88 {'win': 87, 'tie_super_over': 1} {2023: 22, 2024: 22, 2025: 22, 2026: 22} 0.398
20642 {1: 10852, 2: 9790} inn2 missing target: 0 reduced-overs innings: 0
window ok: True
stale columns: []


In [13]:
import numpy as np
import pandas as pd
from scipy import optimize, special, stats

R_MAX = 300

class ParScore:
    def __init__(self, mean='dls', noise='nb'):
        assert mean in ('dls', 'poly') and noise in ('nb', 'poisson')
        self.mean, self.noise = mean, noise

    def _mu(self, theta, u, w):
        if self.mean == 'dls':
            # E[R] = (r(w)/b(w)) * (1 - exp(-b(w)*u)), r = runs/over at the start, b = curvature
            # log r(w) = a0 - exp(t1)*w   -> the scoring rate can only fall as wickets fall
            # log b(w) = c0 + c1*w
            a0, t1, c0, c1 = theta[:4]
            r = np.exp(a0 - np.exp(t1) * w)
            b = np.exp(c0 + c1 * w)
            mu = (r / b) * (-np.expm1(-b * u))          # expm1: accurate when b*u is small
        else:
            # earlier draft: log mu = log(u) + g0 + g1*w + g2*w^2 + g3*(u/20) + g4*w*(u/20)
            x = u / 20.0
            X = np.column_stack([np.ones_like(u), w, w ** 2, x, w * x])
            mu = np.exp(np.log(np.maximum(u, 1e-9)) + X @ theta[:5])
        return np.where((u <= 0) | (w >= 10), 0.0, mu)

    def _start(self, rate):
        base = [np.log(rate), np.log(0.05), np.log(0.05), 0.1] if self.mean == 'dls' else [np.log(rate), 0, 0, 0, 0]
        return np.array(base + ([np.log(0.05)] if self.noise == 'nb' else []))

    def _nll(self, theta, u, w, r):
        mu = np.maximum(self._mu(theta, u, w), 1e-9)
        if self.noise == 'poisson':
            return -(r * np.log(mu) - mu - special.gammaln(r + 1)).sum()
        k = 1.0 / np.exp(theta[-1])                     # k = 1/alpha; Var = mu + alpha*mu^2
        return -(special.gammaln(r + k) - special.gammaln(k) - special.gammaln(r + 1)
                 + k * np.log(k / (k + mu)) + r * np.log(mu / (k + mu))).sum()

    def fit(self, balls):
        inn1 = balls[balls['innings'] == 1]
        r = (inn1.groupby('match_id')['cum_runs'].transform('max') - inn1['cum_runs']).values.astype(float)
        u = inn1['balls_remaining'].values / 6.0
        w = inn1['cum_wickets'].values.astype(float)
        keep = (u > 0) & (w < 10)                       # no resources left -> R = 0, no information
        u, w, r = u[keep], w[keep], r[keep]
        res = optimize.minimize(self._nll, self._start(r.sum() / u.sum()), args=(u, w, r), method='L-BFGS-B')
        if not res.success:
            raise RuntimeError(f'par-score fit failed ({self.mean}/{self.noise}): {res.message}')
        self.theta_ = res.x
        self.alpha_ = float(np.exp(res.x[-1])) if self.noise == 'nb' else 0.0
        per_match = inn1.groupby('match_id').agg(total=('cum_runs', 'max'), y=('y', 'first'))
        t, y = per_match['total'].values.astype(float), per_match['y'].values.astype(float)
        self.t_mean_, self.t_sd_ = t.mean(), t.std()
        z = (t - self.t_mean_) / self.t_sd_
        self.ab_ = optimize.minimize(lambda ab: np.sum(np.logaddexp(0, ab[0] + ab[1] * z) - y * (ab[0] + ab[1] * z)),
                                         np.zeros(2), method='BFGS').x
        return self

    def expected_runs(self, balls_left, wickets_lost):
        return self._mu(self.theta_, np.asarray(balls_left, float) / 6, np.asarray(wickets_lost, float))

    def pmf(self, k, mu):
        return stats.poisson.pmf(k, mu) if self.noise == 'poisson' else stats.nbinom.pmf(k, 1 / self.alpha_, 1 / (1 + self.alpha_ * mu))

    def _cdf(self, k, mu):
        return stats.poisson.cdf(k, mu) if self.noise == 'poisson' else stats.nbinom.cdf(k, 1 / self.alpha_, 1 / (1 + self.alpha_ * mu))

    def predict(self, df):
        mu = np.maximum(self.expected_runs(df['balls_remaining'].values, df['cum_wickets'].values), 1e-12)
        live = mu > 1e-9
        out = np.empty(len(df))
        m1 = (df['innings'] == 1).values
        if m1.any():
            k = np.arange(R_MAX + 1)
            pmf_vals = np.where(live[m1, None], self.pmf(k[None, :], mu[m1, None]), (k[None, :] == 0).astype(float))
            pmf_vals /= pmf_vals.sum(axis=1, keepdims=True)
            totals = df['cum_runs'].values[m1][:, None] + k[None, :]
            out[m1] = (pmf_vals * special.expit(self.ab_[0] + self.ab_[1] * (totals - self.t_mean_) / self.t_sd_)).sum(axis=1)
        m2 = (df['innings'] == 2).values
        if m2.any():
            need = df['runs_needed'].values[m2]           # tie (R = need-1) -> super over, coin flip
            out[m2] = self._cdf(need - 2, mu[m2]) + 0.5 * self.pmf(need - 1, mu[m2])
        term = terminal_prob(df).values                   # decided chases: exact 0 / 0.5 / 1
        return np.where(np.isnan(term), out, term)

PARSCORE_VARIANTS = {'parscore': ('dls', 'nb'), 'parscore_poisson': ('dls', 'poisson'), 'parscore_poly': ('poly', 'nb')}


def parscore_oof(df):
    """Out-of-fold predictions for every variant, on the same season folds as the live models."""
    parts = []
    for train_seasons, test_season in season_folds(df['season'].unique()):
        tr, te = df[df['season'].isin(train_seasons)], df[df['season'] == test_season]
        out = te[['match_id', 'innings', 'seq']].copy()
        for name, (mean, noise) in PARSCORE_VARIANTS.items():
            out[name] = ParScore(mean, noise).fit(tr).predict(te)
        parts.append(out)
    return pd.concat(parts, ignore_index=True)


In [16]:
# Step 5: metrics with match-level bootstrap (the ONLY definition of these functions)
import numpy as np
import pandas as pd

EPS = 1e-6


def log_loss_vec(y, p):
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    y = np.asarray(y, float)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def brier_vec(y, p):
    return (np.asarray(p, float) - np.asarray(y, float)) ** 2


LOSSES = {'log_loss': log_loss_vec, 'brier': brier_vec}


def bootstrap_table(df, models, loss='log_loss', weighting='match', reference=None,
                    n_boot=2000, seed=0, alpha=0.05):
    """Point estimate and percentile CI per model, plus paired differences vs `reference`.
    The match is the resampling unit (its ~230 ball rows share one label), and the SAME
    resampled matches are used for every model, so differences are paired.
    weighting='match': average within each match, then across matches (every match counts once).
    weighting='ball' : average over all rows (long matches count more)."""
    f = LOSSES[loss]
    codes, uniq = pd.factorize(df['match_id'].values)
    n = len(uniq)
    counts = np.bincount(codes, minlength=n).astype(float)
    sums = np.column_stack([np.bincount(codes, weights=f(df['y'].values, df[m].values), minlength=n) for m in models])

    def stat(s, c):
        return (s / c[..., None]).mean(axis=-2) if weighting == 'match' else s.sum(axis=-2) / c.sum(axis=-1)[..., None]

    point = stat(sums, counts)
    idx = np.random.default_rng(seed).integers(0, n, size=(n_boot, n))
    boot = np.stack([stat(sums[i], counts[i]) for i in idx])
    lo, hi = np.quantile(boot, [alpha / 2, 1 - alpha / 2], axis=0)
    rows = []
    for k, m in enumerate(models):
        r = {'model': m, loss: point[k], 'ci_low': lo[k], 'ci_high': hi[k]}
        if reference is not None:
            d = boot[:, k] - boot[:, models.index(reference)]
            r['delta'] = point[k] - point[models.index(reference)]
            r['delta_ci_low'], r['delta_ci_high'] = np.quantile(d, [alpha / 2, 1 - alpha / 2])
        rows.append(r)
    out = pd.DataFrame(rows)
    out.attrs.update(n_matches=n, n_rows=int(counts.sum()))
    return out


def paired_delta(df, a, b, n_boot=2000, seed=0):
    """Per-match log-loss of a minus b, with paired bootstrap CI. Negative = a is better."""
    r = bootstrap_table(df, [a, b], reference=b, n_boot=n_boot, seed=seed).iloc[0]
    return {'a': a, 'b': b, 'delta': float(r['delta']), 'ci_low': float(r['delta_ci_low']), 'ci_high': float(r['delta_ci_high'])}


def reliability(y, p, n_bins=10):
    y, p = np.asarray(y, float), np.asarray(p, float)
    bins = np.clip((p * n_bins).astype(int), 0, n_bins - 1)
    rows = [{'bin': b, 'mean_pred': p[bins == b].mean(), 'observed': y[bins == b].mean(), 'n': int((bins == b).sum())}
            for b in range(n_bins) if (bins == b).any()]
    return pd.DataFrame(rows)


def ece(y, p, n_bins=10):
    r = reliability(y, p, n_bins)
    return float((r['n'] * (r['mean_pred'] - r['observed']).abs()).sum() / r['n'].sum())


def phase_of(over_num):
    return np.where(over_num <= 5, 'powerplay', np.where(over_num <= 15, 'middle', 'death'))

In [17]:
# Step 4 checks + the first held-out table (uses the step 5 metrics cell)
ps = ParScore('dls', 'nb').fit(ball_df)
a0, t1, c0, c1 = ps.theta_[:4]
print('alpha %.3f | death-overs rate r(0) %.2f, falls %.1f%%/wicket | b(0) %.4f'
      % (ps.alpha_, np.exp(a0), 100 * (1 - np.exp(-np.exp(t1))), np.exp(c0)))
print('expected runs: full innings', ps.expected_runs([120], [0]).round(1),
      '| 60 left 3 down', ps.expected_runs([60], [3]).round(1))
print('monotone in wickets:', np.all(np.diff(ps.expected_runs([60] * 10, list(range(10)))) < 0))

# out-of-fold predictions, one row per delivery: the key is seq, never over_num
KEYS = ['match_id', 'innings', 'seq']
oof = live_result['oof'].merge(parscore_oof(ball_df), on=KEYS, how='left', validate='1:1')
print('rows', len(oof), '| expected', sum(len(ball_df[ball_df.season == s]) for _, s in season_folds(ball_df.season.unique())),
      '| missing', oof[list(PARSCORE_VARIANTS)].isna().sum().sum())

LIVE_MODELS = ['base_rate', 'parscore', 'parscore_poisson', 'parscore_poly', 'logistic', 'rf']
t = bootstrap_table(oof, LIVE_MODELS, 'log_loss', 'match', 'base_rate')
print(t.round(3).to_string(index=False), t.attrs)
for a, b in (('parscore_poisson', 'parscore'), ('parscore_poly', 'parscore'), ('logistic', 'parscore'), ('rf', 'logistic')):
    d = paired_delta(oof, a, b)
    print(f"{a} - {b}: {d['delta']:+.3f} [{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]")

alpha 0.044 | death-overs rate r(0) 15.03, falls 7.0%/wicket | b(0) 0.0715
expected runs: full innings [160.] | 60 left 3 down [84.4]
monotone in wickets: True
rows 15643 | expected 15643 | missing 0
           model  log_loss  ci_low  ci_high  delta  delta_ci_low  delta_ci_high
       base_rate     0.703   0.633    0.770  0.000         0.000          0.000
        parscore     0.528   0.428    0.633 -0.175        -0.286         -0.059
parscore_poisson     0.663   0.472    0.874 -0.040        -0.235          0.178
   parscore_poly     0.527   0.427    0.633 -0.175        -0.286         -0.061
        logistic     0.575   0.453    0.717 -0.128        -0.243          0.002
              rf     0.652   0.512    0.814 -0.050        -0.185          0.101 {'n_matches': 66, 'n_rows': 15643}
parscore_poisson - parscore: +0.135 [+0.039, +0.244]
parscore_poly - parscore: -0.001 [-0.003, +0.002]
logistic - parscore: +0.047 [-0.023, +0.118]
rf - logistic: +0.078 [+0.029, +0.133]


In [18]:
# Calibration sensitivity analysis (D-004): would an honestly fitted calibrator have helped? Not deployed.
import numpy as np
from scipy import optimize
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression


def _logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fit_calibrator(method, p, y):
    """Returns a function p -> calibrated p."""
    if method == 'platt':
        m = LogisticRegression(C=1e6, max_iter=1000).fit(_logit(p)[:, None], y)
        return lambda q: m.predict_proba(_logit(q)[:, None])[:, 1]
    if method == 'temperature':
        s = np.exp(optimize.minimize_scalar(
            lambda t: log_loss_vec(y, 1 / (1 + np.exp(-_logit(p) * np.exp(t)))).mean(),
            bounds=(-3, 3), method='bounded').x)
        return lambda q: 1 / (1 + np.exp(-_logit(q) * s))
    if method == 'isotonic':
        m = IsotonicRegression(y_min=0.001, y_max=0.999, out_of_bounds='clip').fit(p, y)
        return m.predict
    raise ValueError(method)


def calibrated_from_earlier_folds(oof, col, method):
    """For each test season S, fit the calibrator per innings on the out-of-fold predictions
    of ALL EARLIER test seasons (out-of-sample, pooled, never season S itself), then apply it
    to S. Seasons with no earlier out-of-fold predictions get NaN. Rows where the match is
    already decided (oof['decided']) are not used for fitting and keep their exact 0 / 0.5 / 1."""
    out = np.full(len(oof), np.nan)
    for s in sorted(oof['season'].unique()):
        for inn in (1, 2):
            fit_rows = (oof['season'] < s) & (oof['innings'] == inn) & ~oof['decided']
            app_rows = ((oof['season'] == s) & (oof['innings'] == inn) & ~oof['decided']).values
            if fit_rows.sum() == 0:
                continue
            cal = fit_calibrator(method, oof.loc[fit_rows, col].values, oof.loc[fit_rows, 'y'].values)
            out[app_rows] = cal(oof.loc[app_rows, col].values)
    has_cal = np.isin(oof['season'].values, oof['season'].values[~np.isnan(out)])
    keep = oof['decided'].values & has_cal
    out[keep] = oof[col].values[keep]
    return out

In [19]:

oof = oof.loc[:, ~oof.columns.str.startswith('decided')]
oof = oof.merge(ball_df[KEYS].assign(decided=terminal_prob(ball_df).notna().values), on=KEYS, how='left', validate='1:1')
CALIBRATED = ('logistic', 'rf', 'parscore')
for method in ('platt', 'temperature', 'isotonic'):
    for col in CALIBRATED:
        oof[f'{col}+{method}'] = calibrated_from_earlier_folds(oof, col, method)
cal_seasons = [int(s) for s in sorted(oof.season.unique())[1:]]
# Reset index so that row indices inside bootstrap_table align perfectly with group indices
cal_df = oof[oof.season.isin(cal_seasons)].reset_index(drop=True)
print('calibration test seasons:', cal_seasons, '| matches:', cal_df.match_id.nunique(),
      '| non-NaN in first season (must be 0):', oof[~oof.season.isin(cal_seasons)]['logistic+platt'].notna().sum())
for col in CALIBRATED:
    t = bootstrap_table(cal_df, [col] + [f'{col}+{m}' for m in ('platt', 'temperature', 'isotonic')], 'log_loss', 'match', col)
    print(t[['model', 'log_loss', 'ci_low', 'ci_high', 'delta', 'delta_ci_low', 'delta_ci_high']].round(3).to_string(index=False))

calibration test seasons: [2025, 2026] | matches: 44 | non-NaN in first season (must be 0): 0
               model  log_loss  ci_low  ci_high  delta  delta_ci_low  delta_ci_high
            logistic     0.483   0.405    0.564  0.000         0.000          0.000
      logistic+platt     0.635   0.532    0.737  0.152         0.072          0.236
logistic+temperature     0.532   0.495    0.570  0.049         0.003          0.095
   logistic+isotonic     0.661   0.551    0.781  0.178         0.089          0.273
         model  log_loss  ci_low  ci_high  delta  delta_ci_low  delta_ci_high
            rf     0.549   0.450    0.666  0.000         0.000          0.000
      rf+platt     0.647   0.544    0.760  0.097         0.020          0.174
rf+temperature     0.554   0.508    0.602  0.005        -0.066          0.065
   rf+isotonic     0.683   0.554    0.844  0.134         0.043          0.231
               model  log_loss  ci_low  ci_high  delta  delta_ci_low  delta_ci_high
            

In [28]:
import pickle
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# Ensure BASE is defined even if the setup cell wasn't run yet
BASE = globals().get('BASE', '.')

DEPLOY_RULE = 'parscore_baseline'
REPORT_DIR = f'{BASE}/reports/stage1'
LEGACY_PKL = f'{BASE}/legacy/models/live_model.pkl'
N_BOOT, SEED = 2000, 0
LABELS = {'base_rate': 'Base rate (constant)', 'parscore': 'Par-score: DLS curve + negative binomial',
          'parscore_poisson': 'Par-score: DLS curve + Poisson (reference)',
          'parscore_poly': 'Par-score: polynomial + negative binomial (reference)',
          'logistic': 'Logistic, per innings', 'rf': 'Random forest, per innings',
          'legacy_v0': 'v0 random forest as deployed (in-sample)'}
def fmt_ci(v, lo, hi):
    return f'{v:.3f} [{lo:.3f}, {hi:.3f}]'
def headline_table(oof,models):
  ll = bootstrap_table(oof, models, 'log_loss', 'match', 'base_rate',N_BOOT, SEED)
  br = bootstrap_table(oof,models,'brier','match', 'base_rate',N_BOOT, SEED)
  pb = bootstrap_table(oof,models,'log_loss','ball',None,N_BOOT, SEED)
  table = pd.DataFrame({
      'Model': [LABELS[m] for m in models],
      'Log-loss per match [95% CI]': [fmt_ci(*v) for v in ll[['log_loss', 'ci_low', 'ci_high']].values],
      'Δ vs base rate [95% CI]': [fmt_ci(*r) for r in ll[['delta', 'delta_ci_low', 'delta_ci_high']].values],
      'Brier per match [95% CI]': [fmt_ci(*r) for r in br[['brier', 'ci_low', 'ci_high']].values],
      'Log-loss per ball [95% CI]': [fmt_ci(*r) for r in pb[['log_loss', 'ci_low', 'ci_high']].values],
      'ECE': [f'{ece(oof.y, oof[m]):.3f}' for m in models]})
  return table, ll
def phase_table(oof,models):
  phase = phase_of(oof['over_num'].values)
  rows = []
  for inn in (1,2):
    for ph in ('powerplay','middle','death'):
      sub = oof[(oof.innings == inn)&(phase==ph)]
      tbl = bootstrap_table(sub,models,'log_loss','ball',None,N_BOOT, SEED)
      rows.append({'Innings': inn, 'Phase': ph, 'Matches': sub.match_id.nunique(), 'Rows': len(sub),
                         **{m: fmt_ci(*r) for m, r in zip(models, tbl[['log_loss', 'ci_low', 'ci_high']].values)}})
  return pd.DataFrame(rows)
def calibration_table(oof,families = ('logistic','rf','parscore'),methods = ('platt','temperature','isotonic')):
  cal_df = oof[oof.season.isin(sorted(oof.season.unique())[1:])]
  rows = []
  for col in families:
    tbl = bootstrap_table(cal_df, [col] + [f'{col}+{m}' for m in methods], 'log_loss', 'match', col, N_BOOT, SEED)
    rows += [{'Model': r['model'], 'Log-loss per match [95% CI]': fmt_ci(r['log_loss'], r['ci_low'], r['ci_high']),
                  'Δ vs uncalibrated [95% CI]': fmt_ci(r['delta'], r['delta_ci_low'], r['delta_ci_high'])}
                 for _, r in tbl.iterrows()]
  return pd.DataFrame(rows), cal_df.match_id.nunique()

def season_rate_test(match_df):
  sc = match_df.groupby('season')['y'].agg(['sum','count'])
  chi2, p, df_, _ = chi2_contingency(np.column_stack([sc['sum'], sc['count'] - sc['sum']]), correction=False)
  text = ('Team batting first won ' + ', '.join(f"{int(r['sum'])}/{int(r['count'])} ({s})" for s, r in sc.iterrows())
            + f'; chi-square test of equal rates: χ² = {chi2:.2f}, df = {df_}, p = {p:.3f}.')
  return text, {'chi2': float(chi2), 'df': int(df_), 'p': float(p)}

In [29]:
def decide_live(oof,rule):
  if rule == 'parscore_baseline':
    tests = {m: paired_delta(oof,m,'parscore',N_BOOT, SEED) for m in ('logistic','rf')}
    winners = [m for m,d in tests.items() if d['ci_high']<0]
    return (min(winners, key=lambda m: tests[m]['delta']) if winners else 'parscore'), tests
  if rule == 'logistic_vs_rf':
    tests = {'rf': paired_delta(oof, 'rf', 'logistic', N_BOOT, SEED)}
    return ('rf' if tests['rf']['ci_high'] < 0 else 'logistic'), tests
  raise ValueError(rule)
def decide_prematch(pm_oof):
    d = paired_delta(pm_oof, 'logistic', 'base_rate', N_BOOT, SEED)
    return ('logistic' if d['ci_high'] < 0 else 'base_rate'), d



In [30]:
def reliability_plot(oof, models, path):
    matplotlib.use('Agg')
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharey=True)
    for ax, inn in zip(axes, (1, 2)):
        sub = oof[oof.innings == inn]
        ax.plot([0, 1], [0, 1], ls='--', c='grey', lw=1, label='perfect calibration')
        for m in models:
            r = reliability(sub.y, sub[m])
            ax.plot(r.mean_pred, r.observed, marker='o', ms=3, label=LABELS[m])
        ax.set_title(f'Innings {inn}: held-out seasons')
        ax.set_xlabel('Predicted P(team batting first wins)')
        ax.grid(alpha=0.3)
    axes[0].set_ylabel('Observed frequency')
    axes[1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)


def legacy_v0_predict(balls, pkl_path):
    """v0 exactly as the v0 API used it (raw, unscaled inputs), converted to the bat-first label."""
    with open(pkl_path, 'rb') as f:
        bundle = pickle.load(f)
    b = balls.sort_values(KEYS)
    r36 = b.groupby(['match_id', 'innings'])['delivery_runs'].transform(lambda s: s.rolling(36, min_periods=1).sum())
    bf = b['legal_balls'].clip(upper=120)
    orem = (120 - bf).clip(lower=0) / 6
    crr = np.where(bf > 0, b['cum_runs'] / (bf / 6).replace(0, np.nan), 0.0)
    tgt = np.where(b['innings'] == 2, b['target'], 0.0)
    need = tgt - b['cum_runs']
    rrr = np.nan_to_num(np.where(orem > 0, need / orem.replace(0, np.nan), 0.0))
    over = bf // 6
    X = pd.DataFrame({'innings': b['innings'], 'cum_runs': b['cum_runs'], 'cum_wickets': b['cum_wickets'],
                      'wickets_left': 10 - b['cum_wickets'], 'balls_faced': bf, 'balls_remaining': orem * 6,
                      'overs_remaining': orem, 'current_rr': crr, 'required_rr': rrr, 'rr_diff': rrr - crr,
                      'runs_last_6_overs': r36, 'is_powerplay': (over < 6).astype(int),
                      'is_middle_overs': ((over >= 6) & (over < 16)).astype(int), 'is_death_overs': (over >= 16).astype(int),
                      'target': tgt, 'runs_needed': need})
    p_bat = bundle['model'].predict_proba(X[bundle['features']].fillna(0).values)[:, 1]
    return b[KEYS].assign(legacy_v0=np.where(b['innings'].values == 1, p_bat, 1 - p_bat))



In [31]:
os.makedirs(REPORT_DIR, exist_ok=True)
if os.path.exists(LEGACY_PKL) and 'legacy_v0' not in oof:
    oof = oof.merge(legacy_v0_predict(ball_df[ball_df.match_id.isin(oof.match_id)], LEGACY_PKL),
                    on=KEYS, how='left', validate='1:1')
MAIN = [m for m in LABELS if m in oof]
FOCUS = [m for m in ('base_rate', 'parscore', 'logistic', 'rf', 'legacy_v0') if m in oof]

prematch_table, team_history = build_match_features(match_df)
prematch_result = train_prematch_model(prematch_table)
pm_oof = prematch_result['oof']
live_tbl,ll = headline_table(oof,MAIN)
ph_tbl= phase_table(oof,FOCUS)
cal_tbl ,n_cal = calibration_table(oof)
# Fix: season_rate_test should be run on match_df instead of oof
season_text,season_stats = season_rate_test(match_df)
LIVE_FAMILY,live_tests = decide_live(oof,DEPLOY_RULE)
PREMATCH_FAMILY, pm_test = decide_prematch(pm_oof)
# Fix: Change 'match' to 'match_id'
pm_ll = bootstrap_table(pm_oof, ['base_rate', 'logistic'], 'log_loss', 'match', 'base_rate', N_BOOT, SEED)
pm_tbl = pd.DataFrame({'Model': ['Base rate', 'Pre-match logistic (form, head-to-head, toss)'],
                       # Fix: Double bracket indexing for DataFrame columns
                       'Log-loss [95% CI]':[fmt_ci(*r) for r in pm_ll[['log_loss', 'ci_low', 'ci_high']].values],
                       'Δ vs base rate [95% CI]': [fmt_ci(*r) for r in pm_ll[['delta', 'delta_ci_low', 'delta_ci_high']].values],
                       'Accuracy': [f"{((pm_oof[m] > 0.5) == pm_oof.y).mean():.3f}" for m in ('base_rate', 'logistic')]})
reliability_plot(oof, [m for m in FOCUS if m != 'base_rate'], f'{REPORT_DIR}/reliability.png')

fold: train [2023] -> test 2024 (22 matches)
fold: train [2023, 2024] -> test 2025 (22 matches)
fold: train [2023, 2024, 2025] -> test 2026 (22 matches)


In [32]:
decision_lines = [f'Rule: `{DEPLOY_RULE}`.'] + \
    [f"{d['a']} − {d['b']}: {d['delta']:+.4f} [{d['ci_low']:+.4f}, {d['ci_high']:+.4f}]" for d in live_tests.values()] + \
    [f'→ live model deployed: **{LIVE_FAMILY}**.',
     f"Pre-match: logistic − base rate {pm_test['delta']:+.4f} [{pm_test['ci_low']:+.4f}, {pm_test['ci_high']:+.4f}] "
     f'→ serve **{PREMATCH_FAMILY}**.']
folds = '; '.join(f'train {tr} → test {te}' for tr, te in season_folds(ball_df.season.unique()))
report = '\n\n'.join([
    '# Stage 1 results',
    f"Snapshot `{os.path.basename(zip_path)}` (sha256 `{SNAPSHOT_SHA[:16]}…`): {len(match_df)} matches. Folds: {folds}. "
    f'Test set: {oof.match_id.nunique()} matches, {len(oof)} ball rows. 95% CIs: percentile bootstrap over matches, '
    f'{N_BOOT} resamples, paired across models.',
    'Label: P(team batting first wins); super-over ties count for the super-over winner. ' + season_text,
    '## Live models (both innings, held-out seasons)', live_tbl.to_markdown(index=False),
    '## Per innings and phase (log-loss per ball [95% CI])', ph_tbl.to_markdown(index=False),
    f'## Calibration sensitivity ({n_cal} matches; calibrators fitted on earlier seasons only; not deployed)',
    cal_tbl.to_markdown(index=False),
    '## Pre-match model (post-toss)', pm_tbl.to_markdown(index=False),
    '## Deployment decision', '\n'.join(f'- {l}' for l in decision_lines),
    '_The v0 row is in-sample and receives raw, unscaled inputs exactly as the v0 API sent them._' if 'legacy_v0' in oof else ''])

with open(f'{REPORT_DIR}/results.md', 'w') as f:
    f.write(report + '\n')
ll.to_csv(f'{REPORT_DIR}/results_live.csv', index=False)
ph_tbl.to_csv(f'{REPORT_DIR}/results_phase.csv', index=False)
cal_tbl.to_csv(f'{REPORT_DIR}/results_calibration.csv', index=False)
pm_ll.to_csv(f'{REPORT_DIR}/results_prematch.csv', index=False)
oof.to_csv(f'{REPORT_DIR}/test_predictions.csv.gz', index=False)
DECISION = {'rule': DEPLOY_RULE, 'live': LIVE_FAMILY, 'live_tests': live_tests,
            'prematch': PREMATCH_FAMILY, 'prematch_test': pm_test, 'season_rates_chi2': season_stats,
            'snapshot': os.path.basename(zip_path), 'snapshot_sha256': SNAPSHOT_SHA, 'n_boot': N_BOOT, 'seed': SEED}
with open(f'{REPORT_DIR}/decision.json', 'w') as f:
    json.dump(DECISION, f, indent=2)
print(report)

# Stage 1 results

Snapshot `wpl_json_20261025.zip` (sha256 `533e7937056b47f1…`): 88 matches. Folds: train [2023] → test 2024; train [2023, 2024] → test 2025; train [2023, 2024, 2025] → test 2026. Test set: 66 matches, 15643 ball rows. 95% CIs: percentile bootstrap over matches, 2000 resamples, paired across models.

Label: P(team batting first wins); super-over ties count for the super-over winner. Team batting first won 7/22 (2023), 11/22 (2024), 7/22 (2025), 10/22 (2026); chi-square test of equal rates: χ² = 2.42, df = 3, p = 0.490.

## Live models (both innings, held-out seasons)

| Model                                                 | Log-loss per match [95% CI]   | Δ vs base rate [95% CI]   | Brier per match [95% CI]   | Log-loss per ball [95% CI]   |   ECE |
|:------------------------------------------------------|:------------------------------|:--------------------------|:---------------------------|:-----------------------------|------:|
| Base rate (constant)              

In [27]:
# Step 8: save exactly what was evaluated, in a form the app can load, with full provenance
import platform
from datetime import datetime, timezone
from importlib.metadata import version

MODEL_VERSION = 'v1'
MODEL_DIR = f'{BASE}/models'
os.makedirs(MODEL_DIR, exist_ok=True)

META = {
    'model_version': MODEL_VERSION,
    'trained_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'python': platform.python_version(),
    'libraries': {k: version(k) for k in ('numpy', 'pandas', 'scikit-learn', 'scipy')},
    'snapshot': os.path.basename(zip_path), 'snapshot_sha256': SNAPSHOT_SHA,
    'n_matches': int(len(match_df)),
    'matches_by_season': {str(k): int(v) for k, v in match_df.groupby('season').size().items()},
    'label': 'P(team batting first wins); super-over ties -> super-over winner; no-results excluded',
    'deploy_rule': DEPLOY_RULE,
}


# ---- plain-number exports (JSON: readable, diffable, no pickle, no version lock-in) -------------
def parscore_to_params(ps):
    return {'mean': ps.mean, 'noise': ps.noise, 'theta': [float(x) for x in ps.theta_], 'alpha': float(ps.alpha_),
            'link': {'a': float(ps.ab_[0]), 'c': float(ps.ab_[1]), 't_mean': float(ps.t_mean_), 't_sd': float(ps.t_sd_)},
            'r_max': R_MAX}


def parscore_from_params(d):
    ps = ParScore(d['mean'], d['noise'])
    ps.theta_, ps.alpha_ = np.array(d['theta']), d['alpha']
    ps.ab_ = np.array([d['link']['a'], d['link']['c']])
    ps.t_mean_, ps.t_sd_ = d['link']['t_mean'], d['link']['t_sd']
    return ps


def logistic_to_params(pipe, features):
    sc, lr = pipe.named_steps['scaler'], pipe.named_steps['lr']
    return {'features': list(features), 'mean': sc.mean_.tolist(), 'scale': sc.scale_.tolist(),
            'coef': lr.coef_[0].tolist(), 'intercept': float(lr.intercept_[0])}


def logistic_from_params(d, X):
    z = ((np.asarray(X, float) - np.array(d['mean'])) / np.array(d['scale'])) @ np.array(d['coef']) + d['intercept']
    return 1 / (1 + np.exp(-z))


# ---- 1. live model: refit the chosen family on ALL seasons --------------------------------
if LIVE_FAMILY == 'parscore':
    final_live = ParScore('dls', 'nb').fit(ball_df)
    live_bundle = {'kind': 'live', 'family': 'parscore', 'params': parscore_to_params(final_live)}
    offline_p = final_live.predict(ball_df)
    reloaded_p = parscore_from_params(json.loads(json.dumps(live_bundle['params']))).predict(ball_df)
elif LIVE_FAMILY == 'logistic':
    models = live_result['final']['logistic']
    live_bundle = {'kind': 'live', 'family': 'logistic',
                   'params': {str(i): logistic_to_params(models[i], FEATURES[i]) for i in (1, 2)}}
    offline_p = predict_per_innings(models, ball_df)
    rp = np.empty(len(ball_df))
    for i in (1, 2):
        m = (ball_df.innings == i).values
        rp[m] = logistic_from_params(live_bundle['params'][str(i)], ball_df.loc[m, FEATURES[i]])
    term = terminal_prob(ball_df).values
    reloaded_p = np.where(np.isnan(term), rp, term)
else:
    raise NotImplementedError('RF would need joblib (pickle) and identical library versions in the app')
live_bundle.update({'window': WINDOW, 'rrr_cap': 36.0, 'metadata': META})
with open(f'{MODEL_DIR}/live_{MODEL_VERSION}.json', 'w') as f:
    json.dump(live_bundle, f, indent=1)
print('live round-trip max |diff|:', float(np.max(np.abs(offline_p - reloaded_p))))

# ---- 2. pre-match model ------------------------------------------------------------------------
pm_bundle = {'kind': 'prematch', 'family': PREMATCH_FAMILY, 'p_base': prematch_result['p_base'],
             'features': PREMATCH_FEATURES, 'history': team_history.to_dict(), 'teams': team_history.teams,
             'metadata': META}
if PREMATCH_FAMILY == 'logistic':
    pm_bundle['params'] = logistic_to_params(prematch_result['final'], PREMATCH_FEATURES)
with open(f'{MODEL_DIR}/prematch_{MODEL_VERSION}.json', 'w') as f:
    json.dump(pm_bundle, f, indent=1)

# ---- 3. parity fixture: states + the probabilities the NOTEBOOK gives -----------------------
# The app must reproduce these exactly from the raw ball history (tested in VS Code, step 9/10).
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(ball_df), size=200, replace=False)
fixture = []
for i in sample_idx:
    r = ball_df.iloc[i]
    hist = ball_df[(ball_df.match_id == r.match_id) & (ball_df.innings == r.innings) & (ball_df.seq <= r.seq)].sort_values('seq')
    wk = np.diff(np.concatenate([[0], hist.cum_wickets.values]))
    fixture.append({'innings': int(r.innings), 'max_balls': int(r.max_balls),
                    'target': None if r.innings == 1 else int(r.target),
                    'balls': [[int(a), int(b), int(c)] for a, b, c in zip(hist.delivery_runs, hist.is_legal, wk)],
                    'expected_bat_first_prob': float(offline_p[i])})
with open(f'{MODEL_DIR}/parity_fixture.json', 'w') as f:
    json.dump({'model_version': MODEL_VERSION, 'snapshot_sha256': SNAPSHOT_SHA, 'cases': fixture}, f)

# ---- 4. model card (human-readable provenance) ------------------------------------------------
card = {**META,
        'live': {'family': LIVE_FAMILY, 'trained_on': 'all seasons in the snapshot; no post-hoc calibrator',
                 'tests': DECISION['live_tests']},
        'prematch': {'family': PREMATCH_FAMILY, 'features': PREMATCH_FEATURES, 'teams': team_history.teams,
                     'test': DECISION['prematch_test']},
        'evaluation': f'{REPORT_DIR}/results.md',
        'limitations': ['No team, player or venue identity in the live model (stage 3).',
                        'WPL only: about 22 matches per season, so intervals are wide.',
                        'Pre-match team history is frozen at the snapshot date until the next retrain.',
                        'A rain-shortened first innings is modelled with the scheduled overs.']}
with open(f'{MODEL_DIR}/model_card.json', 'w') as f:
    json.dump(card, f, indent=2)
print('saved:', sorted(os.listdir(MODEL_DIR)), '| live:', LIVE_FAMILY, '| pre-match:', PREMATCH_FAMILY)

live round-trip max |diff|: 0.0
saved: ['live_v1.json', 'model_card.json', 'parity_fixture.json', 'prematch_v1.json'] | live: parscore | pre-match: base_rate


In [42]:
from google.colab import userdata
import getpass

try:
    t = userdata.get('GITHUB_TOKEN')
except userdata.SecretNotFoundError:
    print("GITHUB_TOKEN was not found in Colab Secrets (the key icon in the left sidebar).")
    t = getpass.getpass("Please enter your GITHUB_TOKEN manually: ")

!git clone https://{t}@github.com/Riks1/WPL-Final.git repo
%cd repo
!git checkout -b stage1

Cloning into 'repo'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 44 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 5.80 MiB | 8.09 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/repo
Switched to a new branch 'stage1'


In [44]:
!mkdir -p legacy/models models data/snapshots reports/stage1 notebooks docs
!git mv live_model.pkl prematch_model.pkl legacy/models/

fatal: bad source, source=live_model.pkl, destination=legacy/models/live_model.pkl


In [45]:
!unzip -o /content/step9_vscode_files.zip -d /content/repo

Archive:  /content/step9_vscode_files.zip
  inflating: /content/repo/requirements.txt  
   creating: /content/repo/app/
   creating: /content/repo/app/services/
  inflating: /content/repo/app/services/prematch.py  
  inflating: /content/repo/app/services/live.py  
  inflating: /content/repo/app/services/model_loader.py  
  inflating: /content/repo/app/config.py  
  inflating: /content/repo/app/__init__.py  
  inflating: /content/repo/app/schemas.py  
   creating: /content/repo/app/routes/
  inflating: /content/repo/app/routes/cricapi.py  
  inflating: /content/repo/app/routes/health.py  
   creating: /content/repo/templates/
  inflating: /content/repo/templates/index.html  
   creating: /content/repo/tests/
  inflating: /content/repo/tests/test_app.py  
  inflating: /content/repo/tests/test_parity_fixture.py  
  inflating: /content/repo/tests/conftest.py  
  inflating: /content/repo/run.py    
   creating: /content/repo/wpl/
  inflating: /content/repo/wpl/__init__.py  
  inflating: /co

In [47]:
%cd /content/repo
!git status --short

/content/repo
?? app/
?? requirements.txt
?? run.py
?? templates/
?? tests/
?? wpl/


In [51]:
!cp /content/models/{live_v1.json,prematch_v1.json,model_card.json,parity_fixture.json} /content/repo/models/
!mkdir -p /content/drive/MyDrive/wpl
!cp -r /content/models /content/reports /content/data /content/drive/MyDrive/wpl/
!ls /content/repo/models

live_v1.json  model_card.json  parity_fixture.json  prematch_v1.json


In [58]:
!pip install -q --only-binary=:all: Flask==3.0.0 Flask-SQLAlchemy==3.1.1 marshmallow==3.22.0 python-dotenv==1.0.1 Werkzeug==3.0.1 gunicorn==23.0.0 pytest==8.3.3

In [63]:
!unzip -n /content/app_complete.zip -d /content/repo

Archive:  /content/app_complete.zip
  inflating: /content/repo/app/errors.py  
 extracting: /content/repo/app/services/__init__.py  
  inflating: /content/repo/app/extensions.py  
  inflating: /content/repo/app/models.py  
  inflating: /content/repo/app/logging_config.py  
  inflating: /content/repo/app/routes/main.py  
  inflating: /content/repo/app/routes/history.py  
 extracting: /content/repo/app/routes/__init__.py  
  inflating: /content/repo/app/routes/predict.py  
 extracting: /content/repo/tests/__init__.py  


In [64]:
!sed -i 's/^numpy==.*/numpy==2.1.3/' /content/repo/requirements.txt

In [60]:
!sed -i 's/^numpy==.*/numpy==2.1.3/' /content/repo/requirements.txt
!grep -E "numpy|pandas|scipy" /content/repo/requirements.txt

numpy==2.1.3
pandas==2.2.3
scipy==1.14.1


In [65]:
%cd /content/repo
!ls app models
!python -m pytest tests -q --durations=5

/content/repo
app:
config.py  extensions.py  logging_config.py  __pycache__  schemas.py
errors.py  __init__.py	  models.py	     routes	  services

models:
live_v1.json  model_card.json  parity_fixture.json  prematch_v1.json
........................                                                 [100%]
============================= slowest 5 durations ==============================
1.84s call     tests/test_parity_fixture.py::test_serving_code_reproduces_notebook
0.53s call     tests/test_parity_fixture.py::test_api_reproduces_notebook
0.12s setup    tests/test_app.py::test_health
0.03s call     tests/test_app.py::test_live_separates_won_and_lost_chases
0.03s call     tests/test_app.py::test_live_more_runs_helps_the_chaser
24 passed in 2.95s


In [66]:
%%writefile /content/repo/requirements-dev.txt
-r requirements.txt
pytest==8.3.3

Writing /content/repo/requirements-dev.txt


In [67]:
%%writefile /content/repo/Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
RUN mkdir -p instance logs
ENV PYTHONUNBUFFERED=1 PORT=7860
EXPOSE 7860
CMD gunicorn --bind 0.0.0.0:${PORT} --workers 2 --timeout 60 run:app

Writing /content/repo/Dockerfile


In [68]:
%%writefile /content/repo/.dockerignore
.git
.github
__pycache__
*.pyc
.env
tests
.pytest_cache
instance
logs
legacy
data
notebooks
reports/stage1/test_predictions.csv.gz

Writing /content/repo/.dockerignore


In [69]:
!mkdir -p /content/repo/.github/workflows
%%writefile /content/repo/.github/workflows/ci.yml
name: CI
on:
  push:
    branches: [ main, stage1 ]
  pull_request:
    branches: [ main ]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -r requirements-dev.txt
      - run: pytest -v

SyntaxError: invalid syntax (4232441798.py, line 4)

In [ ]:
!ls "/content/drive/MyDrive/Colab Notebooks/"
!cp "/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb" /content/repo/notebooks/stage1_training.ipynb

In [ ]:
!mkdir -p reports/stage1 data/snapshots
!cp /content/reports/stage1/* reports/stage1/
!cp /content/data/snapshots/wpl_json_20261025.zip /content/data/snapshots/wpl_json_20261025.manifest.json data/snapshots/
!ls reports/stage1 data/snapshots